# SustainaBuild: Peak-Aware Energy Management with Reinforcement Learning

**COMP6008 Reinforcement Learning - Group 10**
Dasith Gamage, Priyandan Francis, Ushan Gunarathne

This notebook contains all of the code for our project. It builds a simulator of a small solar-powered office building in Perth and trains three RL agents (DQN, REINFORCE and A2C). Each agent picks a cooling level and a battery action every hour. All three are compared against a rule-based controller and a short two-step lookahead controller, using cost, comfort, peak grid import and battery throughput.

**How to run (Google Colab)**
1. Open the notebook in Colab and choose `Runtime > Run all`. A GPU is not needed, because the networks are small and everything runs on the CPU.
2. The weather data is read from `Data/perth_hourly_weather_2021_2024.csv` if that file exists. If it doesn't (for example on a fresh Colab runtime), the notebook downloads the same extract from the Open-Meteo archive API and saves it.
3. `RETRAIN = False` (Section 1) reuses the training runs saved in `Results/` from our full Colab run, so the whole notebook finishes in a few minutes. Setting `RETRAIN = True` trains everything from scratch, which took a bit over an hour on a Colab CPU runtime (learning-rate sweep about 20 min, final runs about 31 min, plus the second sweep). `QUICK_RUN = True` gives a short smoke test with tiny budgets.
4. Trained models (`.pkl`), result tables (`.csv`) and figures (`.png`) are written to `Results/`. On Colab this folder is deleted when the runtime disconnects, so by default (`SAVE_TO_DRIVE = True`) the notebook mounts Google Drive and uses `MyDrive/RL_Assignment/` for `Data/` and `Results/`. Colab will ask you to allow Drive access the first time. If `RETRAIN = False` and saved results already exist, the training cells load them instead of training again.

**Contents**
1. Setup
2. Weather data
3. Building model and parameters
4. The SustainaBuild environment (MDP)
5. Environment checks
6. Conventional controllers (rule-based and two-step lookahead)
7. Shared training and evaluation utilities
8. Agent 1: DQN
9. Agent 2: REINFORCE
10. Agent 3: A2C
11. Hyperparameter tuning
12. Final training runs
13. Experiment E1: unseen test days
14. Experiment E2: shifted occupancy schedule
15. Experiment E3: weather stress tests
16. 24-hour demonstration day
17. Summary of findings
18. References

## 1. Setup
**Author: Dasith Gamage**

Imports, global switches and random seeds. `QUICK_RUN` reduces every training budget so the notebook can be checked end to end quickly.

In [ ]:
import importlib.util, subprocess, sys

# gymnasium is not always pre-installed on Colab
if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium"], check=True)

import os, time, copy, json, random, pickle
from dataclasses import dataclass, field, replace, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
from gymnasium import spaces
from IPython.display import display

QUICK_RUN = False     # True -> tiny budgets, only for checking the code runs
RETRAIN = False       # False -> reuse the saved runs in Results/ (from our full Colab run); True -> train everything again

SAVE_TO_DRIVE = True   # on Colab: keep Data/ and Results/ in Google Drive so they survive the session
DRIVE_FOLDER = "/content/drive/MyDrive/RL_Assignment"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
PROJECT_DIR = "."
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = DRIVE_FOLDER
DATA_DIR = next((os.path.join(PROJECT_DIR, n) for n in ("data", "Data")
                 if os.path.isdir(os.path.join(PROJECT_DIR, n))), os.path.join(PROJECT_DIR, "data"))
RESULTS_DIR = next((os.path.join(PROJECT_DIR, n) for n in ("Results", "results")
                    if os.path.isdir(os.path.join(PROJECT_DIR, n))), os.path.join(PROJECT_DIR, "results"))
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
DEVICE = torch.device("cpu")   # small MLPs: CPU is faster than moving tiny batches to a GPU

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(0)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})
print("project folder:", os.path.abspath(PROJECT_DIR))
print("torch", torch.__version__, "| gymnasium", gym.__version__, "| numpy", np.__version__)

## 2. Weather data
**Author: Dasith Gamage**

The building is driven by real hourly weather for Perth (Bentley, lat -32.0, lon 115.9) from the Open-Meteo historical archive [6]. We use two variables:

| Column | Meaning | Unit |
|---|---|---|
| `temp_c` | air temperature at 2 m, instantaneous value at the start of the hour | °C |
| `ghi_wm2` | global horizontal irradiance (shortwave radiation) | W/m² |

Open-Meteo reports shortwave radiation as the mean over the *preceding* hour, so the value stamped 10:00 is the 09:00-10:00 average. We shift it back by one row so that index `t` always means the interval `[t, t+1)`, which is the same hour the controller is acting in.

The building only has cooling, so we keep the warm season (October to April) and drop May to September. The split is chronological so that no test-period weather leaks into training:

| Split | Period | Use |
|---|---|---|
| Train | Oct 2021 - Apr 2022, Oct 2022 - Apr 2023 | episodes for learning |
| Validation | Oct 2023 - Dec 2023 | hyperparameter choice and checkpoint selection |
| Test | Jan 2024 - Apr 2024 | held-out evaluation (E1-E3), never used for any choice |

In [ ]:
WEATHER_CSV = os.path.join(DATA_DIR, "perth_hourly_weather_2021_2024.csv")

def download_weather(path):
    """Fetch the frozen extract from Open-Meteo (only used if the CSV is missing)."""
    import requests
    url = ("https://archive-api.open-meteo.com/v1/archive?latitude=-32.005&longitude=115.894"
           "&start_date=2021-10-01&end_date=2024-04-30"
           "&hourly=temperature_2m,shortwave_radiation&timezone=Australia%2FPerth")
    js = requests.get(url, timeout=60).json()
    out = pd.DataFrame(js["hourly"])
    out.columns = ["time", "temp_c", "ghi_wm2"]
    out.to_csv(path, index=False)
    print("downloaded", len(out), "rows ->", path)

if not os.path.exists(WEATHER_CSV):
    download_weather(WEATHER_CSV)

weather = pd.read_csv(WEATHER_CSV, parse_dates=["time"])
# interval convention: ghi[t] should be the average over [t, t+1)
weather["ghi_wm2"] = weather["ghi_wm2"].shift(-1).fillna(0.0)
weather["date"] = weather["time"].dt.normalize()
print(weather.shape, "| missing values:", int(weather.isna().sum().sum()))
weather.head()

Next we reshape the hourly series into one row per day (24 values each) and keep the previous day's weather next to it. The previous day is only used by the lookahead controller, as a simple persistence forecast ("tomorrow at 3 pm looks like today at 3 pm").

In [ ]:
@dataclass
class DaySet:
    dates: pd.DatetimeIndex
    temp: np.ndarray        # (n_days, 24) outdoor temperature, degC
    ghi: np.ndarray         # (n_days, 24) irradiance, W/m2
    prev_temp: np.ndarray   # previous day's temperature (persistence forecast only)
    prev_ghi: np.ndarray
    weekday: np.ndarray     # True for Mon-Fri

    def __len__(self):
        return len(self.dates)

    def subset(self, idx):
        return DaySet(self.dates[idx], self.temp[idx], self.ghi[idx],
                      self.prev_temp[idx], self.prev_ghi[idx], self.weekday[idx])

def build_dayset(df):
    groups = [(d, g) for d, g in df.groupby("date") if len(g) == 24]
    dates = pd.DatetimeIndex([d for d, _ in groups])
    temp = np.stack([g["temp_c"].to_numpy() for _, g in groups])
    ghi = np.stack([g["ghi_wm2"].to_numpy() for _, g in groups])
    # previous calendar day; the first day of the file has none so it is dropped below
    prev_temp = np.vstack([np.full((1, 24), np.nan), temp[:-1]])
    prev_ghi = np.vstack([np.full((1, 24), np.nan), ghi[:-1]])
    consecutive = np.r_[False, np.diff(dates.values).astype("timedelta64[D]").astype(int) == 1]
    keep = consecutive & dates.month.isin([10, 11, 12, 1, 2, 3, 4])
    ds = DaySet(dates, temp, ghi, prev_temp, prev_ghi, dates.dayofweek.values < 5)
    return ds.subset(np.where(keep)[0])

all_days = build_dayset(weather)
d = all_days.dates
train_days = all_days.subset(np.where(d < "2023-07-01")[0])
val_days = all_days.subset(np.where((d >= "2023-10-01") & (d < "2024-01-01"))[0])
test_days = all_days.subset(np.where(d >= "2024-01-01")[0])

for name, ds in [("train", train_days), ("validation", val_days), ("test", test_days)]:
    print(f"{name:<11} {len(ds):>4} days  ({ds.dates[0].date()} to {ds.dates[-1].date()}), "
          f"weekdays={ds.weekday.sum()}, mean daily max temp={ds.temp.max(1).mean():.1f} C")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ds, name in [(train_days, "train"), (val_days, "validation"), (test_days, "test")]:
    axes[0].hist(ds.temp.max(1), bins=25, alpha=0.5, density=True, label=name)
    axes[1].plot(range(24), ds.temp.mean(0), label=name)
    axes[2].plot(range(24), ds.ghi.mean(0), label=name)
axes[0].set(xlabel="daily max temperature (°C)", ylabel="density", title="Hot-day distribution")
axes[1].set(xlabel="hour of day", ylabel="°C", title="Mean outdoor temperature")
axes[2].set(xlabel="hour of day", ylabel="W/m²", title="Mean irradiance")
axes[0].legend()
plt.tight_layout(); plt.show()

The test period (January to April 2024) is a bit hotter on average than validation (October to December), but the training data covers the same summer months from earlier years, so the agents have seen similar conditions. The 44.5 °C day on 19 Feb 2024 is the hottest day in the file, and it falls in the test set.

## 3. Building model and parameters
**Author: Dasith Gamage**

The building is a single-storey office of about 500 m² with rooftop PV, one battery, one air-conditioning unit and a grid connection. Every parameter is in one dataclass so that each controller uses exactly the same building, and the stress tests can change a copy of it.

**Thermal model (two-node RC, after Bacher and Madsen [2]).** $T_a$ is the indoor air temperature (including furniture) and $T_m$ is the temperature of the building mass (slab, walls). With heat flows in kW:

$$q_{oa} = \frac{T_{out} - T_a}{R_{oa}}, \qquad q_{ma} = \frac{T_m - T_a}{R_{am}}$$

$$C_a \frac{dT_a}{dt} = q_{oa} + q_{ma} + Q_{int} - \kappa P_{cool}, \qquad C_m \frac{dT_m}{dt} = -q_{ma}$$

$Q_{int}$ is the sum of people heat, the share of the plug/lighting load that ends up as heat, and solar gain through glazing: $Q_{int} = g_{occ}\,o_t + f_{heat} P_{load,t} + A_{sol} G_t/1000$.

In the proposal this was written as a single Euler step with $\Delta t = 1$ h (Eq. 1-2). When we checked numerical stability, a single hourly step is unstable for our parameters: the condition $\frac{\Delta t}{C_a}\left(\frac{1}{R_{oa}} + \frac{1}{R_{am}}\right) < 1$ evaluates to 2.6. So we keep the same equations but integrate them with 12 sub-steps of 5 minutes inside each hour, holding weather and actions constant. Section 5 shows this.

**Parameter choices.** No calibration data from a real building was available, so the values are typical of a small commercial building and were sanity-checked with the free-floating and thermostat runs in Section 5:

| Symbol | Value | Comment |
|---|---|---|
| $C_a$ | 4 kWh/°C | air + furniture |
| $C_m$ | 40 kWh/°C | concrete slab and internal walls |
| $R_{oa}$ | 2.0 °C/kW | envelope + infiltration, UA ≈ 0.5 kW/°C |
| $R_{am}$ | 0.1 °C/kW | air-to-surface coupling (~2000 m² surfaces) |
| $P_{cool}^{max}$, $\kappa$ | 12 kW electric, COP 3 | 36 kW of cooling at full power |
| PV | 40 kWp, 0.85 derate, -0.4 %/°C | cell temp ≈ $T_{out}$ + 0.03 $G$ |
| Battery | 60 kWh, 20 kW, 95 % / 95 %, SoC 10-95 % | starts each day at 50 % |
| Load | 4 kW base, 14 kW when occupied | weekdays 08:00-18:00 |
| Tariff | 0.15 / 0.25 / 0.45 \$/kWh | off-peak 21-07, shoulder 07-15, peak 15-21 |
| Comfort band | 22-25 °C | only counted in occupied hours |
| Peak limit $P_{limit}$ | 15 kW | modelled operating limit, not a utility charge |

In [ ]:
@dataclass
class BuildingConfig:
    # thermal model
    C_a: float = 4.0            # kWh/degC
    C_m: float = 40.0           # kWh/degC
    R_oa: float = 2.0           # degC/kW
    R_am: float = 0.1           # degC/kW
    solar_aperture: float = 12.0    # effective glazing area (m2) -> kW of gain per kW/m2
    occupant_gain: float = 5.0      # kW from people when fully occupied
    load_heat_frac: float = 0.8     # share of electrical load released as heat inside
    n_substeps: int = 12            # 5-minute integration inside each hour
    dt: float = 1.0                 # hours per decision step
    # HVAC
    p_cool_max: float = 12.0    # kW electric
    cop: float = 3.0
    # PV
    pv_kwp: float = 40.0
    pv_derate: float = 0.85
    pv_temp_coef: float = -0.004
    # battery
    batt_capacity: float = 60.0     # kWh
    batt_power: float = 20.0        # kW rated, both directions
    eta_c: float = 0.95
    eta_d: float = 0.95
    soc_min: float = 0.10
    soc_max: float = 0.95
    soc_init: float = 0.50
    # load and occupancy
    base_load: float = 4.0
    occupied_load: float = 14.0
    open_hour: int = 8
    close_hour: int = 18
    schedule_shift: int = 0         # E2: shift occupancy and load by this many hours
    # tariff ($/kWh)
    price_offpeak: float = 0.15
    price_shoulder: float = 0.25
    price_peak: float = 0.45
    # comfort and initial temperatures
    t_low: float = 22.0
    t_high: float = 25.0
    t_air_init: float = 24.0
    t_mass_init: float = 25.0
    # reward weights (fixed on training/validation data before testing)
    peak_limit: float = 15.0        # kW
    lambda_d: float = 2.0           # $ per degree-hour of occupied discomfort
    lambda_p: float = 1.0           # $ per kWh imported above the peak limit
    lambda_b: float = 0.02          # $ per kWh battery throughput (wear proxy)
    p_ref: float = 0.15             # $ per kWh for the terminal battery adjustment (off-peak refill cost)
    # scenario modifiers for stress tests (E3)
    temp_offset: float = 0.0
    solar_scale: float = 1.0

BASE_CFG = BuildingConfig()

COOL_LEVELS = np.array([0.0, 0.5, 1.0])                 # off, half, full
BATT_LEVELS = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])     # full discharge ... full charge
N_ACTIONS = len(COOL_LEVELS) * len(BATT_LEVELS)          # 15

def decode_action(a):
    """joint action index -> (cooling index, battery index). a = 5*cool + batt"""
    a = np.asarray(a)
    return a // len(BATT_LEVELS), a % len(BATT_LEVELS)

ACTION_NAMES = [f"cool={['off','half','full'][c]}, batt={['-full','-half','hold','+half','+full'][b]}"
                for c in range(3) for b in range(5)]
print(N_ACTIONS, "joint actions, e.g.", ACTION_NAMES[0], "|", ACTION_NAMES[7], "|", ACTION_NAMES[14])

The schedules the building operator knows in advance (occupancy, non-cooling load and tariff) are plain functions of the hour and the day type. They are allowed in the observation because they are known ahead of time. Future weather is not.

In [ ]:
def occupancy(cfg, hour, weekday):
    h = (np.asarray(hour) - cfg.schedule_shift) % 24
    occ = (h >= cfg.open_hour) & (h < cfg.close_hour) & np.asarray(weekday)
    return occ.astype(float)

def non_cooling_load(cfg, hour, weekday):
    return cfg.base_load + (cfg.occupied_load - cfg.base_load) * occupancy(cfg, hour, weekday)

def tariff(cfg, hour):
    h = np.asarray(hour) % 24
    return np.where((h >= 15) & (h < 21), cfg.price_peak,
                    np.where((h >= 7) & (h < 15), cfg.price_shoulder, cfg.price_offpeak))

def pv_power(cfg, ghi, t_out):
    t_cell = t_out + 0.03 * ghi
    p = cfg.pv_kwp * (ghi / 1000.0) * cfg.pv_derate * (1 + cfg.pv_temp_coef * (t_cell - 25.0))
    return np.maximum(p, 0.0)

### 3.1 One-hour transition function
This function is the whole transition model $p(s_{t+1} \mid s_t, a_t)$ for one hour, following steps (3)-(5) of the hourly timing in the proposal. It is written with numpy broadcasting, so the lookahead controller can call it for 225 action sequences at once. The environment calls it with scalars.

Battery (charge power $P_{ch}$, discharge power $P_{dis}$, stored energy $E$):

$$E_{t+1} = E_t + \eta_c P_{ch}\Delta t - \frac{P_{dis}}{\eta_d}\Delta t$$

The requested power is first clipped so that $E$ stays within $[\text{SoC}_{min}, \text{SoC}_{max}] \cdot E_{cap}$. Discharge is also capped at the building's net demand, so the battery never exports. Exporting would earn nothing in our setup, so it would only waste stored energy.

Grid balance (Eq. 3): $P_{grid} = P_{load} + P_{cool} + P_{ch} - P_{dis} - P_{PV}$. Import is $\max(0, P_{grid})$. A negative $P_{grid}$ is surplus PV, which is curtailed (no feed-in credit) and logged.

In [ ]:
def building_step(cfg, Ta, Tm, E, cool_idx, batt_idx, t_out, ghi, p_load, occ):
    """Advance the building by one hour. Works on scalars or numpy arrays (broadcast)."""
    dt = cfg.dt
    p_cool = COOL_LEVELS[cool_idx] * cfg.p_cool_max
    p_req = BATT_LEVELS[batt_idx] * cfg.batt_power           # +ve = charge, -ve = discharge
    p_pv = pv_power(cfg, ghi, t_out)

    # --- (3) enforce battery limits ---
    e_min, e_max = cfg.soc_min * cfg.batt_capacity, cfg.soc_max * cfg.batt_capacity
    p_ch_room = np.maximum(0.0, (e_max - E) / (cfg.eta_c * dt))     # max charge that still fits
    p_dis_room = np.maximum(0.0, (E - e_min) * cfg.eta_d / dt)      # max discharge available
    net_demand = np.maximum(0.0, p_load + p_cool - p_pv)            # no battery export
    p_ch = np.minimum(np.maximum(p_req, 0.0), p_ch_room)
    p_dis = np.minimum(np.minimum(np.maximum(-p_req, 0.0), p_dis_room), net_demand)

    # --- (4) battery energy and RC thermal update ---
    E_next = E + cfg.eta_c * p_ch * dt - p_dis / cfg.eta_d * dt
    q_int = cfg.occupant_gain * occ + cfg.load_heat_frac * p_load + cfg.solar_aperture * ghi / 1000.0
    h = dt / cfg.n_substeps
    Ta_sum = 0.0
    for _ in range(cfg.n_substeps):
        q_oa = (t_out - Ta) / cfg.R_oa
        q_ma = (Tm - Ta) / cfg.R_am
        Ta_new = Ta + h / cfg.C_a * (q_oa + q_ma + q_int - cfg.cop * p_cool)
        Tm = Tm - h / cfg.C_m * q_ma
        Ta = Ta_new
        Ta_sum = Ta_sum + Ta
    Ta_mean = Ta_sum / cfg.n_substeps      # used for comfort during the hour

    # --- (5) grid balance ---
    p_grid = p_load + p_cool + p_ch - p_dis - p_pv
    p_import = np.maximum(p_grid, 0.0)
    p_curtail = np.maximum(-p_grid, 0.0)
    return dict(Ta=Ta, Tm=Tm, E=E_next, Ta_mean=Ta_mean, p_cool=p_cool, p_ch=p_ch, p_dis=p_dis,
                p_pv=p_pv, p_load=p_load, p_import=p_import, p_curtail=p_curtail)

### 3.2 Reward (Eq. 4 and 5 of the proposal)

$$r_t = -\big[\,C^{energy}_t + \lambda_D D_t + \lambda_P X_t + \lambda_B B_t\,\big]$$

- $C^{energy}_t = \text{price}_t \cdot P^{import}_t \cdot \Delta t$ is the electricity bill for the hour ($).
- $D_t = o_t\,[\max(0, \bar T_a - 25) + \max(0, 22 - \bar T_a)]\,\Delta t$ is the occupied discomfort in degree-hours, using the hour-average air temperature $\bar T_a$.
- $X_t = \max(0, P^{import}_t - P_{limit})\,\Delta t$ is the energy imported above the 15 kW limit (kWh).
- $B_t = (P_{ch} + P_{dis})\,\Delta t$ is the battery throughput (kWh).

At the last hour we add the terminal adjustment $-p_{ref}(E_{initial} - E_{final})$, so an agent cannot reduce cost by emptying the battery at the end of the day. Every controller gets the same adjustment.

The weights were fixed before any test-day evaluation. A penalty of $\lambda_D = 2$ \$/°C·h means that being 1 °C too warm for a whole working day (10 °C·h, \$20) costs more than running full cooling all day at shoulder price (12 kW × 10 h × 0.25 = \$30 is the same order). So the agent can't simply switch the AC off to save money. $\lambda_P = 1$ \$/kWh is about twice the peak tariff, which makes shaving the afternoon peak worthwhile. $\lambda_B = 0.02$ \$/kWh is a rough wear cost. It is small enough that useful cycling still pays, but it discourages pointless charge/discharge loops.

**Choosing $p_{ref}$.** Our first value was $p_{ref} = 0.25$ \$/kWh (the shoulder price). With that value the lookahead controller learned to fill the battery overnight at 0.15 \$/kWh and finish the day almost full: each stored kWh was credited at 0.25 × 0.95 ≈ 0.24 $ but only cost 0.15 + 0.02 $. That is an artefact of the terminal adjustment, not useful control. We changed $p_{ref}$ to the off-peak price, 0.15 \$/kWh, which is what it costs to refill the battery overnight. Now buying energy just to hold it at the end of the day loses money, while discharging into the 0.45 \$/kWh peak still pays off.

In [ ]:
def reward_terms(cfg, res, price, occ):
    dt = cfg.dt
    cost = price * res["p_import"] * dt
    discomfort = occ * (np.maximum(0.0, res["Ta_mean"] - cfg.t_high) +
                        np.maximum(0.0, cfg.t_low - res["Ta_mean"])) * dt
    exceed = np.maximum(0.0, res["p_import"] - cfg.peak_limit) * dt
    throughput = (res["p_ch"] + res["p_dis"]) * dt
    reward = -(cost + cfg.lambda_d * discomfort + cfg.lambda_p * exceed + cfg.lambda_b * throughput)
    return reward, dict(cost=cost, discomfort=discomfort, exceed=exceed, throughput=throughput)

## 4. The SustainaBuild environment (MDP)
**Author: Dasith Gamage**

**State.** The underlying state is $s_t = (T_a, T_m, E, t, \text{day})$. The day index fixes the weather for that episode. The agent never sees the day index or any future weather, so from the agent's point of view the weather is the stochastic part of the transition. One simplification here: the PV value in the observation is computed from the irradiance averaged over the coming hour, so the agent effectively knows this hour's solar output when it decides. A real controller would only have the latest meter reading or a short-term forecast. We treat this as near-perfect within-hour information, and every controller gets the same value.

**Observation** (14 values, scaled to roughly [-1, 1] so that one learning rate suits all inputs):

| # | Feature | Scaling | Why it is there |
|---|---|---|---|
| 0 | indoor air temp $T_a$ | (T-24)/5 | comfort right now |
| 1 | mass temp $T_m$ | (T-24)/5 | stored "coolth" (pre-cooling) |
| 2 | battery SoC | 0-1 | stored energy |
| 3 | outdoor temp | (T-25)/10 | heat load |
| 4 | outdoor temp change since last hour | ΔT/5 | is it warming or cooling down |
| 5 | PV output | / kWp | free energy available |
| 6 | non-cooling load | / 20 kW | demand |
| 7 | price now | / peak price | |
| 8 | price next hour | / peak price | known tariff schedule |
| 9 | occupied now | 0/1 | |
| 10 | occupied next hour | 0/1 | known schedule, lets the agent pre-cool |
| 11, 12 | sin, cos of hour | | time of day without a jump at midnight |
| 13 | hours remaining / 24 | | the horizon is finite, so this keeps the problem Markov |

**Actions.** `Discrete(15)`: 3 cooling levels × 5 battery levels, $a = 5c + b$.

**Episode.** 24 hourly steps from midnight. During training a random day is drawn from the training split on every reset. For evaluation the day is passed in with `options={"day": i}`, so every controller sees exactly the same days.

In [ ]:
class SustainaBuildEnv(gym.Env):
    metadata = {"render_modes": []}
    OBS_DIM = 14

    def __init__(self, days, cfg=BASE_CFG, seed=None):
        super().__init__()
        self.days = days
        self.cfg = cfg
        self.observation_space = spaces.Box(-5.0, 5.0, shape=(self.OBS_DIM,), dtype=np.float32)
        self.action_space = spaces.Discrete(N_ACTIONS)
        self.np_random_days = np.random.default_rng(seed)
        self.day = 0
        self.t = 0

    # exogenous inputs for an hour of the current day, with any stress-test modifiers applied
    def _weather(self, hour, previous_day=False):
        temp = self.days.prev_temp if previous_day else self.days.temp
        ghi = self.days.prev_ghi if previous_day else self.days.ghi
        return (temp[self.day, hour] + self.cfg.temp_offset,
                ghi[self.day, hour] * self.cfg.solar_scale)

    def _obs(self):
        c, h = self.cfg, self.t
        wd = self.days.weekday[self.day]
        t_out, ghi = self._weather(h)
        t_prev = self._weather(h - 1)[0] if h > 0 else self._weather(23, previous_day=True)[0]
        obs = np.array([
            (self.Ta - 24.0) / 5.0,
            (self.Tm - 24.0) / 5.0,
            self.E / c.batt_capacity,
            (t_out - 25.0) / 10.0,
            (t_out - t_prev) / 5.0,
            pv_power(c, ghi, t_out) / c.pv_kwp if c.pv_kwp > 0 else 0.0,
            non_cooling_load(c, h, wd) / 20.0,
            tariff(c, h) / c.price_peak,
            tariff(c, h + 1) / c.price_peak,
            occupancy(c, h, wd),
            occupancy(c, h + 1, wd),
            np.sin(2 * np.pi * h / 24),
            np.cos(2 * np.pi * h / 24),
            (24 - h) / 24.0,
        ], dtype=np.float32)
        return obs

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.np_random_days = np.random.default_rng(seed)
        if options is not None and "day" in options:
            self.day = int(options["day"])
        else:
            self.day = int(self.np_random_days.integers(len(self.days)))
        c = self.cfg
        self.t = 0
        self.Ta, self.Tm = c.t_air_init, c.t_mass_init
        self.E = c.soc_init * c.batt_capacity
        self.E_initial = self.E
        return self._obs(), {"date": str(self.days.dates[self.day].date())}

    def step(self, action):
        c, h = self.cfg, self.t
        cool_idx, batt_idx = decode_action(int(action))                       # (2) joint action
        wd = self.days.weekday[self.day]
        t_out, ghi = self._weather(h)
        occ, p_load, price = occupancy(c, h, wd), non_cooling_load(c, h, wd), tariff(c, h)

        res = building_step(c, self.Ta, self.Tm, self.E, cool_idx, batt_idx,   # (3)-(5)
                            t_out, ghi, p_load, occ)
        reward, terms = reward_terms(c, res, price, occ)

        self.Ta, self.Tm, self.E = float(res["Ta"]), float(res["Tm"]), float(res["E"])
        self.t += 1
        terminated = self.t >= 24
        terminal_adj = 0.0
        if terminated:
            terminal_adj = c.p_ref * (self.E_initial - self.E)        # Eq. (5)
            reward -= terminal_adj

        info = dict(hour=h, t_out=t_out, occ=occ, price=price, terminal_adj=terminal_adj,
                    Ta=float(res["Ta_mean"]), Tm=self.Tm, soc=self.E / c.batt_capacity,
                    **{k: float(v) for k, v in res.items() if k.startswith("p_")},
                    **{k: float(v) for k, v in terms.items()})
        obs = self._obs() if not terminated else np.zeros(self.OBS_DIM, dtype=np.float32)
        return obs, float(reward), terminated, False, info

    # ---- information the conventional controllers are allowed to use ----
    def known_schedule(self, hour):
        """occupancy, load and price are scheduled in advance, so they can be looked up"""
        wd = self.days.weekday[self.day]
        return occupancy(self.cfg, hour, wd), non_cooling_load(self.cfg, hour, wd), tariff(self.cfg, hour)

    def current_weather(self):
        """this hour's weather - the same values the observation is built from (see the note in Section 4)"""
        return self._weather(self.t)

    def persistence_forecast(self, hour):
        """forecast for a future hour of today = same hour yesterday (past information only)"""
        return self._weather(hour % 24, previous_day=True)

Quick check that the class follows the Gymnasium API, and a look at one observation.

In [ ]:
from gymnasium.utils.env_checker import check_env
_env = SustainaBuildEnv(train_days, seed=0)
check_env(_env, skip_render_check=True)
o, info = _env.reset(seed=1)
print("reset ->", info)
print(np.round(o, 3))

## 5. Environment checks
**Author: Dasith Gamage**

Before training anything, we check the simulator against what it should do physically. If the environment is wrong, the agents just learn to exploit the bug. The checks follow the list in Section 5 of our proposal.

**Check 1: energy balance and battery bounds under random actions.** For every step, grid import minus curtailment must equal load + cooling + charging - discharging - PV. The change in stored energy must equal $\eta_c P_{ch} - P_{dis}/\eta_d$, and SoC has to stay inside 10-95 %.

In [ ]:
env = SustainaBuildEnv(train_days, seed=123)
rng = np.random.default_rng(123)
max_balance_err, max_batt_err, soc_seen = 0.0, 0.0, []
for ep in range(200):
    env.reset()
    E_prev, done = env.E, False
    while not done:
        _, _, done, _, inf = env.step(rng.integers(N_ACTIONS))
        lhs = inf["p_import"] - inf["p_curtail"]
        rhs = inf["p_load"] + inf["p_cool"] + inf["p_ch"] - inf["p_dis"] - inf["p_pv"]
        max_balance_err = max(max_balance_err, abs(lhs - rhs))
        dE = BASE_CFG.eta_c * inf["p_ch"] - inf["p_dis"] / BASE_CFG.eta_d
        max_batt_err = max(max_batt_err, abs((env.E - E_prev) - dE))
        E_prev = env.E
        soc_seen.append(inf["soc"])
print(f"max grid balance error   : {max_balance_err:.2e} kW")
print(f"max battery energy error : {max_batt_err:.2e} kWh")
print(f"SoC range seen           : {min(soc_seen):.3f} - {max(soc_seen):.3f}  (limits 0.10 - 0.95)")
assert max_balance_err < 1e-9 and max_batt_err < 1e-9
assert min(soc_seen) >= 0.1 - 1e-9 and max(soc_seen) <= 0.95 + 1e-9

**Check 2: cooling direction.** Starting from the same state, more cooling should always give a lower indoor temperature, and the battery action should not change the temperature at all.

In [ ]:
args = dict(Ta=26.0, Tm=26.0, E=30.0, t_out=35.0, ghi=800.0, p_load=14.0, occ=1.0)
for c_idx in range(3):
    temps = [building_step(BASE_CFG, cool_idx=c_idx, batt_idx=b, **args)["Ta"] for b in range(5)]
    print(f"cooling={['off ','half','full'][c_idx]}  Ta after 1 h = {temps[0]:.2f} C   "
          f"(spread over battery actions: {np.ptp(temps):.1e})")

**Check 3: numerical stability and free-floating response.** We simulate the hottest test day with the AC off, using different numbers of sub-steps per hour. With one Euler step per hour (the literal form of Eq. 1-2) the air temperature blows up. With 12 sub-steps it agrees with a 120-sub-step reference to within a few hundredths of a degree, so 12 is enough.

In [ ]:
def free_float(cfg, day_set, day):
    e = SustainaBuildEnv(day_set, cfg)
    e.reset(options={"day": day})
    Ta = []
    for _ in range(24):
        e.step(2)            # a = 5*0 + 2: cooling off, battery hold
        Ta.append(e.Ta)
    return np.array(Ta)

hot_day = int(np.argmax(test_days.temp.max(1)))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
for n in [1, 2, 12, 120]:
    Ta = free_float(replace(BASE_CFG, n_substeps=n), test_days, hot_day)
    ax[0].plot(range(1, 25), Ta, marker="o" if n == 1 else None, ms=3, label=f"{n} sub-steps")
ax[0].set(ylim=(0, 60), xlabel="hour", ylabel="indoor air temp (°C)",
          title=f"No cooling, {test_days.dates[hot_day].date()}")
ax[0].plot(range(24), test_days.temp[hot_day], "k--", lw=1, label="outdoor")
ax[0].legend(fontsize=7)
ref = free_float(replace(BASE_CFG, n_substeps=120), test_days, hot_day)
errs = {n: np.abs(free_float(replace(BASE_CFG, n_substeps=n), test_days, hot_day) - ref).max()
        for n in [4, 6, 12, 24]}
ax[1].bar([str(k) for k in errs], errs.values())
ax[1].set(xlabel="sub-steps per hour", ylabel="max |error| vs 120 sub-steps (°C)",
          title="Integration error")
plt.tight_layout(); plt.show()
print({k: round(v, 4) for k, v in errs.items()})

Without cooling, the indoor temperature on the 44.5 °C day climbs into the low 30s and lags the outdoor peak by a few hours, because of the thermal mass. That is the behaviour we expected, and it shows the problem is non-trivial: pre-cooling the mass in the morning (when PV is available and the price is lower) can reduce the cooling needed in the expensive 15:00-21:00 window.

**Check 4: reward arithmetic by hand.** One step at 16:00 (peak price \$0.45) on a weekday, late afternoon with weak sun, full cooling and the battery asked to charge at half power. We redo the PV output, the bill, the exceedance and the throughput penalty by hand and compare them with the environment.

In [ ]:
cfg = BASE_CFG
res = building_step(cfg, Ta=25.5, Tm=25.0, E=40.0, cool_idx=2, batt_idx=3,
                    t_out=36.0, ghi=100.0, p_load=14.0, occ=1.0)
r_env, terms = reward_terms(cfg, res, price=0.45, occ=1.0)

p_pv = 40 * 0.1 * 0.85 * (1 - 0.004 * (36 + 0.03 * 100 - 25))   # PV by hand
p_ch = min(0.5 * 20, (0.95 * 60 - 40) / 0.95)                    # half of rated power, room left in battery
p_import = 14 + 12 + p_ch - p_pv
cost = 0.45 * p_import
disc = max(0, res["Ta_mean"] - 25) + max(0, 22 - res["Ta_mean"])
r_hand = -(cost + 2.0 * disc + 1.0 * max(0, p_import - 15) + 0.02 * p_ch)
print(f"PV {p_pv:.3f} kW | charge {p_ch:.3f} kW | import {p_import:.3f} kW (env {float(res['p_import']):.3f})")
print(f"bill ${cost:.3f}, exceedance {p_import - 15:.3f} kWh, mean air temp {float(res['Ta_mean']):.2f} C")
print(f"reward by hand {r_hand:.4f}  vs  environment {float(r_env):.4f}")
assert abs(r_hand - r_env) < 1e-9

**Check 5: resets are reproducible and the terminal adjustment is applied.** Two resets on the same day with the same action sequence must give identical returns. Discharging all day must trigger a positive terminal charge.

In [ ]:
def rollout_return(env, day, actions):
    env.reset(options={"day": day})
    total = 0.0
    for a in actions:
        _, r, done, _, inf = env.step(a)
        total += r
    return total, inf["terminal_adj"]

env = SustainaBuildEnv(val_days)
acts = np.random.default_rng(5).integers(N_ACTIONS, size=24)
print("same day twice:", rollout_return(env, 3, acts)[0], rollout_return(env, 3, acts)[0])
print("terminal adjustment after discharging all day: $%.2f" % rollout_return(env, 3, [10] * 24)[1])

**Simulation speed.** How long one step takes sets how many training transitions we can afford.

In [ ]:
env = SustainaBuildEnv(train_days, seed=0)
env.reset()
n, t0 = 20000, time.perf_counter()
for i in range(n):
    _, _, done, _, _ = env.step(i % N_ACTIONS)
    if done:
        env.reset()
step_us = (time.perf_counter() - t0) / n * 1e6
print(f"{step_us:.1f} microseconds per environment step -> {1e6 / step_us:,.0f} steps per second")

## 6. Conventional controllers
**Author: Priyandan Francis**

These are the references the RL agents have to beat. Both start from the same observation as the agents, plus the schedule information the operator would know anyway (occupancy, load and tariff). The lookahead also has two advantages the RL agents do not have: it tests actions with the exact simulator equations, and it uses yesterday's weather as a forecast for the next hour. So it is a model-based comparison point with more information, not an equal-information one. Neither controller sees the realised weather beyond the current hour.

**Rule-based controller.** A scheduled thermostat plus a solar- and tariff-aware battery rule, the kind of logic a building manager could set up without any optimisation:
- Cooling runs only when the building is occupied or will be in the next hour (one hour of pre-cooling). It uses full power above 24 °C, half power above 23 °C, and is off otherwise.
- Battery: if PV exceeds the non-cooling load by more than 5 kW, charge at full power (more than 1 kW, half power). Discharge at full power during the peak tariff (15:00-21:00). Top up from the grid overnight if SoC is below 50 %. Otherwise hold.

In [ ]:
def obs_to_dict(obs, cfg=BASE_CFG):
    """undo the observation scaling (used by the hand-written controllers)"""
    return dict(Ta=obs[0] * 5 + 24, Tm=obs[1] * 5 + 24, soc=obs[2],
                E=obs[2] * cfg.batt_capacity, t_out=obs[3] * 10 + 25, pv=obs[5] * cfg.pv_kwp,
                load=obs[6] * 20, price=obs[7] * cfg.price_peak, occ=obs[9], occ_next=obs[10],
                hour=int(round(24 - obs[13] * 24)))

class RuleBasedController:
    name = "Rule-based"

    def act(self, obs, env=None):
        cfg = env.cfg if env is not None else BASE_CFG
        s = obs_to_dict(obs, cfg)
        # thermostat with one hour of pre-cooling
        if s["occ"] > 0.5 or s["occ_next"] > 0.5:
            cool = 2 if s["Ta"] > 24.0 else (1 if s["Ta"] > 23.0 else 0)
        else:
            cool = 0
        # battery rule
        surplus = s["pv"] - s["load"]
        if surplus > 5.0:
            batt = 4
        elif surplus > 1.0:
            batt = 3
        elif s["price"] >= cfg.price_peak - 1e-6:
            batt = 0
        elif s["hour"] < 7 and s["soc"] < 0.5:
            batt = 3
        else:
            batt = 2
        return int(cool * 5 + batt)

**Two-step lookahead controller.** At every hour it tries all 15 × 15 = 225 action pairs for this hour and the next, simulates them with the same `building_step` function, and runs the first action of the best pair. The current hour uses the measured weather (the same values the agents see in their observation). The next hour uses a persistence forecast taken from yesterday at the same hour. Battery energy left at the end of the two hours is valued at $p_{ref}$, as in the terminal adjustment, otherwise a two-hour horizon would never charge the battery.

This is a simple bounded comparison point. It is not an MPC with a proper forecaster and it is not optimal.

In [ ]:
class LookaheadController:
    name = "Lookahead (2-step)"

    def __init__(self, cfg=BASE_CFG):
        self.cfg = cfg
        a1, a2 = np.meshgrid(np.arange(N_ACTIONS), np.arange(N_ACTIONS), indexing="ij")
        self.a1, self.a2 = a1.ravel(), a2.ravel()               # 225 sequences
        self.c1, self.b1 = decode_action(self.a1)
        self.c2, self.b2 = decode_action(self.a2)

    def act(self, obs, env):
        cfg, h = env.cfg, env.t
        s = obs_to_dict(obs, cfg)
        E0 = s["E"]
        total = np.zeros(len(self.a1))
        # hour t: measured weather
        t_out, ghi = env.current_weather()
        occ, load, price = env.known_schedule(h)
        r1 = building_step(cfg, s["Ta"], s["Tm"], E0, self.c1, self.b1, t_out, ghi, load, occ)
        total += reward_terms(cfg, r1, price, occ)[0]
        E_end = r1["E"]
        if h + 1 < 24:
            # hour t+1: persistence forecast for weather, schedule is known
            t_out2, ghi2 = env.persistence_forecast(h + 1)
            occ2, load2, price2 = env.known_schedule(h + 1)
            r2 = building_step(cfg, r1["Ta"], r1["Tm"], r1["E"], self.c2, self.b2,
                               t_out2, ghi2, load2, occ2)
            total += reward_terms(cfg, r2, price2, occ2)[0]
            E_end = r2["E"]
        total -= cfg.p_ref * (E0 - E_end)        # value of energy left in the battery
        return int(self.a1[np.argmax(total)])

A small helper runs any controller over a set of days and collects per-day metrics and the hourly trace. The RL agents use the same function later, so all controllers are scored in exactly the same way.

In [ ]:
def run_controller(controller, day_set, cfg=BASE_CFG, days=None, keep_trace=False):
    env = SustainaBuildEnv(day_set, cfg)
    days = range(len(day_set)) if days is None else days
    rows, traces, latencies = [], [], []
    for d in days:
        obs, _ = env.reset(options={"day": d})
        done, ret, hours = False, 0.0, []
        while not done:
            t0 = time.perf_counter()
            a = controller.act(obs, env)
            latencies.append(time.perf_counter() - t0)
            obs, r, done, _, inf = env.step(a)
            ret += r
            inf["action"] = a
            hours.append(inf)
        h = pd.DataFrame(hours)
        rows.append(dict(
            date=day_set.dates[d].date(), weekday=bool(day_set.weekday[d]),
            objective=-ret,                                   # daily objective incl. terminal adj ($)
            energy_cost=h["cost"].sum(),
            discomfort_degh=h["discomfort"].sum(),
            violation_hours=int(((h["discomfort"] > 1e-6)).sum()),
            peak_import_kw=h["p_import"].max(),
            exceed_kwh=h["exceed"].sum(),
            exceed_hours=int((h["exceed"] > 1e-6).sum()),
            throughput_kwh=h["throughput"].sum(),
            terminal_adj=h["terminal_adj"].sum(),
            final_soc=h["soc"].iloc[-1],
            final_Ta=env.Ta,
            curtailed_kwh=h["p_curtail"].sum(),
        ))
        if keep_trace:
            h["date"] = day_set.dates[d].date()
            traces.append(h)
    out = pd.DataFrame(rows)
    out.attrs["latency_ms"] = 1000 * float(np.mean(latencies))
    if keep_trace:
        out.attrs["trace"] = pd.concat(traces, ignore_index=True)
    return out

def summarise(df):
    keys = ["objective", "energy_cost", "discomfort_degh", "violation_hours", "peak_import_kw",
            "exceed_kwh", "exceed_hours", "throughput_kwh", "final_soc"]
    return df[keys].mean()

rule = RuleBasedController()
lookahead = LookaheadController()
t0 = time.perf_counter()
base_val = {c.name: run_controller(c, val_days) for c in [rule, lookahead]}
print(f"(took {time.perf_counter() - t0:.1f} s)")
print(pd.DataFrame({k: summarise(v) for k, v in base_val.items()}).round(2))

On the validation days the lookahead controller roughly halves the rule-based objective. The gap comes from two places. It almost never imports above the 15 kW limit, because it discharges the battery into the late-afternoon peak when the next hour looks expensive. It also has a much smaller bill, mostly because it lets the indoor temperature drift up towards 25 °C instead of holding 23-24 °C like the thermostat. The price is slightly more discomfort, since it sometimes overshoots the band edge. These two controllers set the bar for the RL agents: anything above the rule-based line is not useful, and the lookahead is a strong target.

## 7. Shared training and evaluation utilities
**Author: Ushan Gunarathne**

All three agents use the same pieces:
- `mlp` builds the networks. Every network has two hidden layers of 128 units, so differences between agents come from the algorithm and not the model size.
- `evaluate_policy` runs a policy greedily over a fixed set of days and returns the mean daily objective in $ (lower is better). We use it on the validation days for the learning curves and for picking the best checkpoint.
- `TrainLog` stores the training history: the return of every training episode, plus the validation objective every `eval_every` environment steps.

**Reward scaling.** Hourly rewards are in dollars and range from about 0 to -40, so returns over a day can reach -150 or worse. Every agent multiplies the reward by `reward_scale = 0.1` before using it for learning. The value targets and gradients then stay at a sensible size (the same idea as reward clipping in [3], but without changing the ordering of rewards). Results are always reported in unscaled dollars.

**Common budget.** Every agent is trained for the same number of environment transitions. This is how we compare sample efficiency, which matters because each day of simulation stands in for a day of real building operation.

In [ ]:
def mlp(in_dim, out_dim, hidden=(128, 128), act=nn.ReLU):
    layers, d = [], in_dim
    for h in hidden:
        layers += [nn.Linear(d, h), act()]
        d = h
    layers.append(nn.Linear(d, out_dim))
    return nn.Sequential(*layers)

class TorchPolicy:
    """wraps a network so it has the same act(obs, env) interface as the rule-based controllers"""
    def __init__(self, net, name):
        self.net, self.name = net, name

    @torch.no_grad()
    def act(self, obs, env=None):
        x = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
        return int(self.net(x).argmax(1).item())   # greedy: argmax Q, or most likely action

@torch.no_grad()
def evaluate_policy(net, day_set, cfg=BASE_CFG):
    """mean daily objective ($, lower is better) of the greedy policy over every day in day_set.
    All days are simulated in parallel (one env per day) so a forward pass covers the whole batch."""
    envs = [SustainaBuildEnv(day_set, cfg) for _ in range(len(day_set))]
    obs = np.stack([e.reset(options={"day": i})[0] for i, e in enumerate(envs)])
    total = np.zeros(len(envs))
    for _ in range(24):
        acts = net(torch.as_tensor(obs)).argmax(1).numpy()
        for i, e in enumerate(envs):
            o, r, _, _, _ = e.step(acts[i])
            obs[i] = o
            total[i] += r
    return float(-total.mean())

@dataclass
class TrainLog:
    ep_steps: list = field(default_factory=list)     # env steps when each training episode ended
    ep_returns: list = field(default_factory=list)   # unscaled training episode return ($)
    eval_steps: list = field(default_factory=list)
    eval_objective: list = field(default_factory=list)   # validation mean daily objective ($)
    best_objective: float = np.inf
    best_state: dict = None
    wall_time: float = 0.0

    def maybe_eval(self, step, net, every, val_set):
        if step % every == 0:
            obj = evaluate_policy(net, val_set)
            self.eval_steps.append(step)
            self.eval_objective.append(obj)
            if obj < self.best_objective:              # keep the best checkpoint on validation
                self.best_objective = obj
                self.best_state = copy.deepcopy(net.state_dict())

## 8. Agent 1: Deep Q-Network (DQN)
**Author: Priyandan Francis**

**Why DQN.** The action space is small and discrete (15 joint actions) and the observation is continuous, which is exactly the setting DQN was designed for [3]. Being off-policy, it can reuse every stored transition many times through experience replay. That matters here because a "day" of experience would be expensive to collect in a real building. DQN learns $Q(s, a)$ for the joint action, so it can learn how cooling and battery choices interact (for example, run the AC on battery power during the 17:00 peak).

**Update.** For a minibatch of transitions $(s, a, r, s', d)$ sampled from the replay buffer $\mathcal{D}$:

$$y = r + \gamma (1 - d)\max_{a'} Q_{\bar\theta}(s', a'), \qquad \mathcal{L}(\theta) = \mathbb{E}_{\mathcal{D}}\big[\ell_\delta\big(Q_\theta(s, a) - y\big)\big]$$

Here $\bar\theta$ are the target-network weights, copied from $\theta$ every $C$ steps, and $\ell_\delta$ is the Huber loss. With `double=True` the target becomes the Double DQN target $y = r + \gamma(1-d)\,Q_{\bar\theta}\big(s', \arg\max_{a'} Q_\theta(s', a')\big)$ [7]. We test this as a modification in Section 11.

**Pseudocode** (the tags `[D1]`...`[D12]` appear as comments in the code below):
```
D1  initialise Q-network θ, target network θ̄ ← θ, replay buffer D (capacity N)
D2  for step = 1 … T:
D3      ε ← linear decay from ε_start to ε_end over the first fraction of T
D4      with prob. ε pick a random action, else a = argmax_a Q_θ(s, a)
D5      execute a, observe r, s', done
D6      store (s, a, r·scale, s', done) in D
D7      if step ≥ learning_starts and step mod train_freq = 0:
D8          sample a minibatch of B transitions from D
D9          compute target y with the target network (max, or double-DQN argmax)
D10         gradient step on Huber(Q_θ(s,a), y), with gradient-norm clipping
D11     every C steps: θ̄ ← θ
D12     s ← s' (reset the environment if done); every K steps evaluate on validation days
```

In [ ]:
@dataclass
class DQNConfig:
    lr: float = 5e-4
    gamma: float = 0.99
    batch_size: int = 64
    buffer_size: int = 50_000
    learning_starts: int = 1_000
    train_freq: int = 2              # one gradient step every 2 environment steps
    target_update: int = 1_000       # C
    eps_start: float = 1.0
    eps_end: float = 0.05
    eps_decay_frac: float = 0.3      # epsilon reaches eps_end after this fraction of training
    double: bool = False
    hidden: tuple = (128, 128)
    grad_clip: float = 10.0
    reward_scale: float = 0.1

class ReplayBuffer:
    """fixed-size circular buffer stored as numpy arrays"""
    def __init__(self, capacity, obs_dim, rng):
        self.s = np.zeros((capacity, obs_dim), np.float32)
        self.s2 = np.zeros((capacity, obs_dim), np.float32)
        self.a = np.zeros(capacity, np.int64)
        self.r = np.zeros(capacity, np.float32)
        self.d = np.zeros(capacity, np.float32)
        self.capacity, self.size, self.ptr, self.rng = capacity, 0, 0, rng

    def add(self, s, a, r, s2, d):
        i = self.ptr
        self.s[i], self.a[i], self.r[i], self.s2[i], self.d[i] = s, a, r, s2, d
        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, n):
        idx = self.rng.integers(self.size, size=n)
        return (torch.as_tensor(self.s[idx]), torch.as_tensor(self.a[idx]), torch.as_tensor(self.r[idx]),
                torch.as_tensor(self.s2[idx]), torch.as_tensor(self.d[idx]))

def train_dqn(cfg, total_steps, seed, train_set=None, val_set=None, eval_every=5_000):
    train_set = train_days if train_set is None else train_set
    val_set = val_days if val_set is None else val_set
    set_seed(seed)
    rng = np.random.default_rng(seed)
    env = SustainaBuildEnv(train_set, seed=seed)
    obs_dim = env.OBS_DIM

    # [D1] networks and replay buffer
    q_net = mlp(obs_dim, N_ACTIONS, cfg.hidden)
    q_target = copy.deepcopy(q_net)
    opt = torch.optim.Adam(q_net.parameters(), lr=cfg.lr)
    buffer = ReplayBuffer(cfg.buffer_size, obs_dim, rng)
    log = TrainLog()

    obs, _ = env.reset()
    ep_ret, t_start = 0.0, time.perf_counter()
    for step in range(1, total_steps + 1):                                     # [D2]
        # [D3] linear epsilon schedule
        frac = min(1.0, step / (cfg.eps_decay_frac * total_steps))
        eps = cfg.eps_start + frac * (cfg.eps_end - cfg.eps_start)
        # [D4] epsilon-greedy action
        if rng.random() < eps:
            action = int(rng.integers(N_ACTIONS))
        else:
            with torch.no_grad():
                action = int(q_net(torch.as_tensor(obs).unsqueeze(0)).argmax(1).item())
        # [D5] environment step
        obs2, r, done, _, _ = env.step(action)
        ep_ret += r
        # [D6] store transition (scaled reward)
        buffer.add(obs, action, r * cfg.reward_scale, obs2, float(done))

        if step >= cfg.learning_starts and step % cfg.train_freq == 0:          # [D7]
            s, a, rb, s2, d = buffer.sample(cfg.batch_size)                     # [D8]
            with torch.no_grad():                                               # [D9]
                if cfg.double:
                    a_star = q_net(s2).argmax(1, keepdim=True)
                    q_next = q_target(s2).gather(1, a_star).squeeze(1)
                else:
                    q_next = q_target(s2).max(1).values
                y = rb + cfg.gamma * (1.0 - d) * q_next
            q_sa = q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
            loss = F.smooth_l1_loss(q_sa, y)                                    # [D10]
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_net.parameters(), cfg.grad_clip)
            opt.step()

        if step % cfg.target_update == 0:                                       # [D11]
            q_target.load_state_dict(q_net.state_dict())

        # [D12] next state / reset, periodic validation
        obs = obs2
        if done:
            log.ep_steps.append(step); log.ep_returns.append(ep_ret)
            obs, _ = env.reset()
            ep_ret = 0.0
        log.maybe_eval(step, q_net, eval_every, val_set)

    log.wall_time = time.perf_counter() - t_start
    return q_net, log

## 9. Agent 2: REINFORCE (Monte-Carlo policy gradient)
**Author: Dasith Gamage**

**Why REINFORCE.** It is the simplest policy-gradient method [4] and it learns a stochastic policy $\pi_\theta(a \mid s)$ directly, with no value function to bootstrap from. Our episodes are short and always end after 24 steps, so complete-episode returns are cheap to compute and unbiased. That makes REINFORCE a clean policy-gradient reference. We expect it to be the least sample-efficient of the three, because the return of a whole day is a noisy signal for any single hourly decision.

**Update.** For a batch of $K$ episodes, with reward-to-go $G_t = \sum_{k=t}^{23} \gamma^{k-t} r_k$:

$$\nabla_\theta J \approx \frac{1}{K \cdot 24}\sum_{i=1}^{K}\sum_{t=0}^{23} \nabla_\theta \log \pi_\theta(a^i_t \mid s^i_t)\,\big(G^i_t - b^i_t\big) \;+\; \beta\,\nabla_\theta \mathcal{H}\big[\pi_\theta(\cdot \mid s^i_t)\big]$$

**Baseline.** The proposal says we would use a *non-learned* baseline. We use a leave-one-out average over the other episodes in the batch at the same hour: $b^i_t = \frac{1}{K-1}\sum_{j \ne i} G^j_t$. It does not depend on episode $i$'s own actions, so the gradient stays unbiased [4]. Because it is computed per hour, it also removes the large time-of-day trend in $G_t$ (a 06:00 return-to-go is always much more negative than a 22:00 one). We also divide the advantages by their standard deviation. That only rescales the step size and leaves the direction unchanged. The entropy bonus $\beta\mathcal{H}$ stops the policy from collapsing onto one action too early.

**Pseudocode** (tags `[R1]`...`[R8]`):
```
R1  initialise policy network θ
R2  repeat until the transition budget T is used:
R3      sample K episodes with a ~ π_θ(·|s)  (24 steps each)
R4      compute discounted reward-to-go G_t for every step
R5      baseline b_t (leave-one-out mean over the batch at the same hour) or 0
R6      advantage A_t = G_t - b_t, divided by its std
R7      loss = -mean(log π_θ(a_t|s_t) · A_t) - β · mean(entropy)
R8      gradient step on θ (with gradient-norm clipping); evaluate on validation every K_eval steps
```

In [ ]:
@dataclass
class REINFORCEConfig:
    lr: float = 1e-3
    gamma: float = 0.99
    episodes_per_update: int = 8     # K
    baseline: bool = True
    normalise_adv: bool = True
    entropy_coef: float = 0.01       # beta
    hidden: tuple = (128, 128)
    grad_clip: float = 1.0
    reward_scale: float = 0.1

def discounted_returns(rewards, gamma):
    """reward-to-go for each row (episode) of a (K, 24) array"""
    G = np.zeros_like(rewards)
    running = np.zeros(rewards.shape[0])
    for t in reversed(range(rewards.shape[1])):
        running = rewards[:, t] + gamma * running
        G[:, t] = running
    return G

def train_reinforce(cfg, total_steps, seed, train_set=None, val_set=None, eval_every=5_000):
    train_set = train_days if train_set is None else train_set
    val_set = val_days if val_set is None else val_set
    set_seed(seed)
    K = cfg.episodes_per_update
    envs = [SustainaBuildEnv(train_set, seed=seed * 100 + i) for i in range(K)]

    policy = mlp(SustainaBuildEnv.OBS_DIM, N_ACTIONS, cfg.hidden, act=nn.Tanh)   # [R1]
    opt = torch.optim.Adam(policy.parameters(), lr=cfg.lr)
    log = TrainLog()
    step, next_eval, t_start = 0, eval_every, time.perf_counter()

    while step < total_steps:                                                    # [R2]
        # [R3] run K episodes side by side (one env each), sampling from the policy
        obs = np.stack([e.reset()[0] for e in envs])
        obs_buf = np.zeros((K, 24, SustainaBuildEnv.OBS_DIM), np.float32)
        act_buf = np.zeros((K, 24), np.int64)
        rew_buf = np.zeros((K, 24))
        for t in range(24):
            with torch.no_grad():
                probs = torch.softmax(policy(torch.as_tensor(obs)), dim=1)
            acts = torch.multinomial(probs, 1).squeeze(1).numpy()
            obs_buf[:, t], act_buf[:, t] = obs, acts
            for i, e in enumerate(envs):
                o, r, _, _, _ = e.step(acts[i])
                obs[i], rew_buf[i, t] = o, r
        step += 24 * K
        for i in range(K):
            log.ep_steps.append(step); log.ep_returns.append(rew_buf[i].sum())

        # [R4] reward-to-go on scaled rewards
        G = discounted_returns(rew_buf * cfg.reward_scale, cfg.gamma)
        # [R5] leave-one-out baseline at each hour
        if cfg.baseline:
            b = (G.sum(0, keepdims=True) - G) / (K - 1)
        else:
            b = np.zeros_like(G)
        # [R6] advantages
        adv = G - b
        if cfg.normalise_adv:
            adv = adv / (adv.std() + 1e-8)

        # [R7] policy-gradient loss with entropy bonus (recomputed with gradients)
        logits = policy(torch.as_tensor(obs_buf.reshape(-1, SustainaBuildEnv.OBS_DIM)))
        dist = torch.distributions.Categorical(logits=logits)
        logp = dist.log_prob(torch.as_tensor(act_buf.reshape(-1)))
        adv_t = torch.as_tensor(adv.reshape(-1), dtype=torch.float32)
        loss = -(logp * adv_t).mean() - cfg.entropy_coef * dist.entropy().mean()

        # [R8] gradient step and periodic validation
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), cfg.grad_clip)
        opt.step()
        if step >= next_eval:
            log.maybe_eval(next_eval, policy, eval_every, val_set)
            next_eval += eval_every

    log.wall_time = time.perf_counter() - t_start
    return policy, log

## 10. Agent 3: Advantage Actor-Critic (A2C)
**Author: Ushan Gunarathne**

**Why A2C.** A2C sits between the other two. Like REINFORCE it learns a stochastic policy (the actor), but it also learns a state-value function $V_\phi(s)$ (the critic) and uses it in two ways: as a learned baseline, and to bootstrap $n$-step returns, so the policy can be updated every $n$ steps instead of waiting for the end of the day [5]. This lowers the variance of the gradient compared with REINFORCE, at the cost of some bias from $V_\phi$. We run it synchronously over 8 environments with different training days. This is the synchronous A2C variant, not the asynchronous A3C from [5]. The 8 parallel environments also decorrelate the samples in each update.

**Update.** For each of the $N$ environments and each of the $n$ rollout steps:

$$\hat R_t = \sum_{k=0}^{n-t-1}\gamma^k r_{t+k} + \gamma^{n-t}(1-d)\,V_\phi(s_{n}), \qquad A_t = \hat R_t - V_\phi(s_t)$$

$$\mathcal{L} = \underbrace{-\frac{1}{Nn}\sum \log\pi_\theta(a_t \mid s_t)\,A_t}_{\text{actor}} + c_v\underbrace{\frac{1}{Nn}\sum\big(\hat R_t - V_\phi(s_t)\big)^2}_{\text{critic}} - \beta\,\underbrace{\frac{1}{Nn}\sum\mathcal{H}[\pi_\theta(\cdot \mid s_t)]}_{\text{entropy}}$$

In the actor term, $A_t$ is treated as a constant (detached), so the actor gradient does not flow into the critic. Before the actor term, the advantages are normalised to zero mean and unit standard deviation within each batch.

**Rollout length $n$ and entropy.** Our first A2C runs used $n = 8$ and $\beta = 0.01$, which are common defaults. Every one of them stalled at about \$18-19 per day on validation, roughly the rule-based level, whatever the learning rate. Looking at the learned policy, the action probabilities had collapsed to almost one-hot within the first ~20k steps. The policy discharged the battery overnight at the off-peak price and had nothing left for the 15:00-21:00 peak. Discharging gives an immediate reward, and the cost only appears through the value of the stored energy later in the day. With $n = 8$ that value comes from a critic that has not learned it yet, so the bootstrapped advantage favours the myopic action. We therefore use $n = 24$ (each update uses complete days, so the critic only serves as a baseline) and $\beta = 0.03$, and Section 11 compares $n \in \{8, 12, 24\}$ directly.

**Pseudocode** (tags `[A1]`...`[A9]`):
```
A1  initialise actor θ, critic φ, N parallel environments
A2  repeat until the transition budget T is used:
A3      for k = 1 … n: sample a_t ~ π_θ(·|s_t) in every env, step, store (log π, V(s_t), r_t, done, entropy)
A4          reset any environment whose episode ended
A5      bootstrap value V_φ(s_n) for the last state (0 if done)
A6      n-step returns R̂_t backwards from the bootstrap
A7      advantages A_t = R̂_t - V_φ(s_t), normalised over the batch for the actor term
A8      loss = actor + c_v · critic - β · entropy
A9      one gradient step on (θ, φ) with gradient-norm clipping; periodic validation
```

In [ ]:
@dataclass
class A2CConfig:
    lr: float = 7e-4
    gamma: float = 0.99
    n_envs: int = 8                  # N
    n_steps: int = 24                # n (24 = one full day per rollout, see Section 11)
    value_coef: float = 0.5          # c_v
    entropy_coef: float = 0.03       # beta
    hidden: tuple = (128, 128)
    grad_clip: float = 0.5
    normalise_adv: bool = True
    reward_scale: float = 0.1

def train_a2c(cfg, total_steps, seed, train_set=None, val_set=None, eval_every=5_000):
    train_set = train_days if train_set is None else train_set
    val_set = val_days if val_set is None else val_set
    set_seed(seed)
    N, n = cfg.n_envs, cfg.n_steps
    obs_dim = SustainaBuildEnv.OBS_DIM

    # [A1] separate actor and critic networks, one optimiser
    actor = mlp(obs_dim, N_ACTIONS, cfg.hidden, act=nn.Tanh)
    critic = mlp(obs_dim, 1, cfg.hidden, act=nn.Tanh)
    opt = torch.optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=cfg.lr)
    envs = [SustainaBuildEnv(train_set, seed=seed * 100 + i) for i in range(N)]
    obs = np.stack([e.reset()[0] for e in envs])
    ep_ret = np.zeros(N)
    log = TrainLog()
    step, next_eval, t_start = 0, eval_every, time.perf_counter()

    while step < total_steps:                                                   # [A2]
        logps, values, rewards, dones, entropies = [], [], [], [], []
        for k in range(n):                                                      # [A3]
            x = torch.as_tensor(obs)
            dist = torch.distributions.Categorical(logits=actor(x))
            acts = dist.sample()
            logps.append(dist.log_prob(acts))
            entropies.append(dist.entropy())
            values.append(critic(x).squeeze(1))
            r_k, d_k = np.zeros(N), np.zeros(N)
            for i, e in enumerate(envs):
                o, r, done, _, _ = e.step(int(acts[i]))
                ep_ret[i] += r
                r_k[i], d_k[i] = r, float(done)
                if done:                                                        # [A4]
                    log.ep_steps.append(step + (k + 1) * N); log.ep_returns.append(ep_ret[i])
                    ep_ret[i] = 0.0
                    o, _ = e.reset()
                obs[i] = o
            rewards.append(torch.as_tensor(r_k * cfg.reward_scale, dtype=torch.float32))
            dones.append(torch.as_tensor(d_k, dtype=torch.float32))
        step += N * n

        with torch.no_grad():                                                   # [A5]
            R = critic(torch.as_tensor(obs)).squeeze(1)
        returns = []
        for k in reversed(range(n)):                                            # [A6]
            R = rewards[k] + cfg.gamma * R * (1.0 - dones[k])
            returns.append(R)
        returns = torch.stack(returns[::-1])
        values = torch.stack(values)
        adv = returns - values                                                  # [A7]

        adv_actor = adv.detach()
        if cfg.normalise_adv:
            adv_actor = (adv_actor - adv_actor.mean()) / (adv_actor.std() + 1e-8)
        actor_loss = -(torch.stack(logps) * adv_actor).mean()                   # [A8]
        critic_loss = adv.pow(2).mean()
        entropy = torch.stack(entropies).mean()
        loss = actor_loss + cfg.value_coef * critic_loss - cfg.entropy_coef * entropy

        opt.zero_grad()                                                         # [A9]
        loss.backward()
        nn.utils.clip_grad_norm_(list(actor.parameters()) + list(critic.parameters()), cfg.grad_clip)
        opt.step()
        if step >= next_eval:
            log.maybe_eval(next_eval, actor, eval_every, val_set)
            next_eval += eval_every

    log.wall_time = time.perf_counter() - t_start
    return actor, log

**Smoke test.** A very short run of each agent to make sure the code paths work and to time them. This is not a result.

In [ ]:
for fn, c in [(train_dqn, DQNConfig()), (train_reinforce, REINFORCEConfig()), (train_a2c, A2CConfig())]:
    net, lg = fn(c, total_steps=5_000, seed=0, eval_every=5_000)
    print(f"{fn.__name__:<16} {lg.wall_time:6.1f} s for 5k steps | "
          f"validation objective after 5k steps: ${lg.eval_objective[-1]:.2f}")

## 11. Hyperparameter tuning
**Author: Ushan Gunarathne (DQN part: Priyandan Francis, REINFORCE part: Dasith Gamage)**

Tuning follows the same two stages for each agent and only ever looks at the **validation** days:
1. **Learning rate.** Three values, each with `len(TUNE_SEEDS)` seeds and `TUNE_STEPS` transitions. The learning rate was the setting that mattered most in our early trial runs.
2. **One structural setting**, at the best learning rate. These are also the modifications we wanted to test:
   - DQN: standard target vs the Double DQN target [7]
   - REINFORCE: leave-one-out baseline vs no baseline
   - A2C: rollout length $n$ ∈ {8, 12, 24}. With $n = 24$ every update uses full-day returns, so the critic only acts as a baseline. With shorter rollouts it also bootstraps.

The score for a configuration is the validation objective of its best checkpoint, averaged over seeds. Every run's result is stored in `Results/`, so with `RETRAIN = False` the section reloads them instead of training again.

In [ ]:
if QUICK_RUN:
    TUNE_STEPS, FINAL_STEPS, EVAL_EVERY = 10_000, 20_000, 5_000
    TUNE_SEEDS, FINAL_SEEDS = [0], [0, 1]
else:
    TUNE_STEPS, FINAL_STEPS, EVAL_EVERY = 100_000, 300_000, 10_000
    TUNE_SEEDS, FINAL_SEEDS = [0, 1], [0, 1, 2]

TRAINERS = {"DQN": (train_dqn, DQNConfig),
            "REINFORCE": (train_reinforce, REINFORCEConfig),
            "A2C": (train_a2c, A2CConfig)}

def run_cached(tag, agent, cfg, steps, seed):
    """train (or load) one run and return a small dict with the network weights and the log"""
    path = os.path.join(RESULTS_DIR, f"{tag}.pkl")
    if not RETRAIN and os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    fn, _ = TRAINERS[agent]
    net, log = fn(cfg, total_steps=steps, seed=seed, eval_every=EVAL_EVERY)
    out = dict(agent=agent, cfg=cfg, seed=seed, steps=steps, log=log, final_state=net.state_dict())
    with open(path, "wb") as f:
        pickle.dump(out, f)
    return out

def tune(agent, name, grid):
    """grid: list of (label, config). Runs every config for every tuning seed."""
    rows, runs = [], {}
    for label, cfg in grid:
        for seed in TUNE_SEEDS:
            tag = f"tune_{agent}_{name}_{label}_s{seed}".replace(".", "p")
            r = run_cached(tag, agent, cfg, TUNE_STEPS, seed)
            runs[(label, seed)] = r
            rows.append(dict(agent=agent, setting=name, value=label, seed=seed,
                             best_val=r["log"].best_objective, last_val=r["log"].eval_objective[-1],
                             minutes=r["log"].wall_time / 60))
    return pd.DataFrame(rows), runs

LR_GRID = {"DQN": [1e-4, 5e-4, 1e-3], "REINFORCE": [3e-4, 1e-3, 3e-3], "A2C": [3e-4, 7e-4, 2e-3]}

t0 = time.perf_counter()
lr_tables, lr_runs = {}, {}
for agent, lrs in LR_GRID.items():
    _, Cfg = TRAINERS[agent]
    lr_tables[agent], lr_runs[agent] = tune(agent, "lr", [(str(lr), Cfg(lr=lr)) for lr in lrs])
lr_table = pd.concat(lr_tables.values(), ignore_index=True)
print(f"learning-rate sweep finished in {(time.perf_counter() - t0) / 60:.1f} min")
lr_table.groupby(["agent", "value"])[["best_val", "last_val", "minutes"]].mean().round(2)

Validation curves for the learning-rate sweep (mean over the tuning seeds). The dashed lines are the two conventional controllers on the same validation days.

In [ ]:
VAL_RULE = summarise(base_val["Rule-based"])["objective"]
VAL_LOOK = summarise(base_val["Lookahead (2-step)"])["objective"]

def plot_sweep(runs_by_agent, title):
    fig, axes = plt.subplots(1, len(runs_by_agent), figsize=(4.2 * len(runs_by_agent), 3.2), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, (agent, runs) in zip(axes, runs_by_agent.items()):
        labels = sorted({k[0] for k in runs}, key=lambda s: (len(s), s))
        for label in labels:
            curves = np.array([runs[(label, s)]["log"].eval_objective for s in TUNE_SEEDS])
            x = np.array(runs[(label, TUNE_SEEDS[0])]["log"].eval_steps) / 1000
            ax.plot(x, curves.mean(0), label=label)
            if len(TUNE_SEEDS) > 1:
                ax.fill_between(x, curves.min(0), curves.max(0), alpha=0.15)
        ax.axhline(VAL_RULE, ls="--", c="grey", lw=1, label="rule-based")
        ax.axhline(VAL_LOOK, ls=":", c="k", lw=1, label="lookahead")
        ax.set(title=agent, xlabel="environment steps (thousands)")
        ax.legend(fontsize=7)
    axes[0].set_ylabel("validation objective ($/day, lower is better)")
    axes[0].set_ylim(0, 80)
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

plot_sweep(lr_runs, "Learning-rate sweep (shaded = range over seeds)")

In [ ]:
best_lr = {a: float(t.groupby("value")["best_val"].mean().idxmin()) for a, t in lr_tables.items()}
print("best learning rate per agent:", best_lr)

stage2_grid = {
    "DQN": ("double", [("False", DQNConfig(lr=best_lr["DQN"], double=False)),
                       ("True", DQNConfig(lr=best_lr["DQN"], double=True))]),
    "REINFORCE": ("baseline", [("True", REINFORCEConfig(lr=best_lr["REINFORCE"], baseline=True)),
                               ("False", REINFORCEConfig(lr=best_lr["REINFORCE"], baseline=False))]),
    "A2C": ("n_steps", [(str(n), A2CConfig(lr=best_lr["A2C"], n_steps=n)) for n in [8, 12, 24]]),
}
s2_tables, s2_runs = {}, {}
for agent, (name, grid) in stage2_grid.items():
    s2_tables[agent], s2_runs[agent] = tune(agent, name, grid)
s2_table = pd.concat(s2_tables.values(), ignore_index=True)
s2_table.groupby(["agent", "setting", "value"])[["best_val", "last_val", "minutes"]].mean().round(2)

In [ ]:
plot_sweep(s2_runs, "Stage 2: Double DQN / REINFORCE baseline / A2C rollout length")

Final configuration for each agent is the best stage-2 setting (which already uses the best learning rate). The full tuning table is saved to `Results/tuning_table.csv`.

In [ ]:
best_cfg = {}
for agent, (name, grid) in stage2_grid.items():
    scores = s2_tables[agent].groupby("value")["best_val"].mean()
    best_cfg[agent] = dict(grid)[scores.idxmin()]
    print(f"{agent:<10} {name}={scores.idxmin():<6} ->", best_cfg[agent])
pd.concat([lr_table, s2_table]).to_csv(os.path.join(RESULTS_DIR, "tuning_table.csv"), index=False)

## 12. Final training runs
**Author: Ushan Gunarathne**

Each agent is retrained from scratch with its chosen configuration on `len(FINAL_SEEDS)` new seeds, for the same `FINAL_STEPS` transitions each. For every seed we keep the checkpoint with the best validation objective. The test days are still untouched at this point.

In [ ]:
final_runs = {}
t0 = time.perf_counter()
for agent in TRAINERS:
    final_runs[agent] = [run_cached(f"final_{agent}_s{s}", agent, best_cfg[agent], FINAL_STEPS, 100 + s)
                         for s in FINAL_SEEDS]
    mins = [r["log"].wall_time / 60 for r in final_runs[agent]]
    print(f"{agent:<10} {len(mins)} seeds, {np.mean(mins):.1f} min per run")
print(f"total {(time.perf_counter() - t0) / 60:.1f} min")

def smooth(y, w):
    y = np.asarray(y, float)
    if len(y) < w:
        return y
    return np.convolve(y, np.ones(w) / w, mode="valid")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
colors = {"DQN": "C0", "REINFORCE": "C1", "A2C": "C2"}
for agent, runs in final_runs.items():
    # training curve: smoothed episode return, interpolated onto a common step grid
    grid_x = np.linspace(5_000, FINAL_STEPS, 60)
    ys = []
    for r in runs:
        lg = r["log"]
        w = 200 if len(lg.ep_returns) > 400 else 20
        y = smooth(lg.ep_returns, w)
        x = np.asarray(lg.ep_steps[w - 1:]) if len(lg.ep_returns) >= w else np.asarray(lg.ep_steps)
        ys.append(np.interp(grid_x, x, y))
    ys = np.array(ys)
    axes[0].plot(grid_x / 1000, ys.mean(0), c=colors[agent], label=agent)
    axes[0].fill_between(grid_x / 1000, ys.min(0), ys.max(0), color=colors[agent], alpha=0.15)
    # validation curve
    v = np.array([r["log"].eval_objective for r in runs])
    vx = np.array(runs[0]["log"].eval_steps) / 1000
    axes[1].plot(vx, v.mean(0), c=colors[agent], label=agent)
    axes[1].fill_between(vx, v.min(0), v.max(0), color=colors[agent], alpha=0.15)
axes[0].set(xlabel="environment steps (thousands)", ylabel="training episode return ($)",
            title="Training return (moving average, exploration on)")
axes[1].axhline(VAL_RULE, ls="--", c="grey", lw=1, label="rule-based")
axes[1].axhline(VAL_LOOK, ls=":", c="k", lw=1, label="lookahead")
axes[1].set(xlabel="environment steps (thousands)", ylabel="validation objective ($/day)",
            title="Greedy policy on validation days", ylim=(0, 60))
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "learning_curves.png"), dpi=150)
plt.show()

The training return (left) is measured while the agents are still exploring. DQN's ε-greedy and the sampling of the two policy-gradient agents both cost money, so the left plot always looks worse than the greedy validation curve on the right. The right-hand plot is the one to use for comparing agents.

The next table summarises learning quality: best validation objective, how much it varies between seeds, and the first point at which the greedy policy beat the rule-based controller on validation (a rough measure of sample efficiency).

In [ ]:
def first_below(log, level):
    for s, v in zip(log.eval_steps, log.eval_objective):
        if v < level:
            return s
    return np.nan

rows = []
for agent, runs in final_runs.items():
    best = [r["log"].best_objective for r in runs]
    reach = [first_below(r["log"], VAL_RULE) for r in runs]
    rows.append({"agent": agent,
                 "best val objective (mean)": np.mean(best),
                 "std over seeds": np.std(best),
                 "worst seed": np.max(best),
                 "steps to beat rule-based (median)": np.nanmedian(reach) if not np.all(np.isnan(reach)) else np.nan,
                 "seeds that beat rule-based": int(np.sum(~np.isnan(reach))),
                 "train minutes per run": np.mean([r["log"].wall_time / 60 for r in runs])})
learning_table = pd.DataFrame(rows).set_index("agent")
learning_table.round(2)

**Export for the demo app.** For each agent we save the weights of the seed with the best validation objective, its configuration and the building configuration in one file (`Results/trained_policies.pt`). The separate dashboard notebook (`SustainaBuild_Demo_App.ipynb`) loads this file, so it never needs to retrain.

In [ ]:
export = {"building_config": asdict(BASE_CFG), "agents": {}}
for agent, runs in final_runs.items():
    best = int(np.argmin([r["log"].best_objective for r in runs]))
    r = runs[best]
    export["agents"][agent] = {"config": asdict(r["cfg"]), "state_dict": r["log"].best_state,
                               "seed": r["seed"], "val_objective": r["log"].best_objective}
torch.save(export, os.path.join(RESULTS_DIR, "trained_policies.pt"))
print("saved", os.path.join(RESULTS_DIR, "trained_policies.pt"),
      {a: round(v["val_objective"], 2) for a, v in export["agents"].items()})

## 13. Experiment E1: unseen test days
**Author: Ushan Gunarathne**

Every controller now runs on the 121 held-out test days (January to April 2024) under the original office schedule. For the RL agents we evaluate the best validation checkpoint of every seed separately. The table reports the mean over days, then mean ± standard deviation across seeds. Decision latency is the average wall-clock time of one `act()` call.

In [ ]:
def make_net(agent, cfg):
    act = nn.ReLU if agent == "DQN" else nn.Tanh
    return mlp(SustainaBuildEnv.OBS_DIM, N_ACTIONS, cfg.hidden, act=act)

def trained_policies(agent):
    pols = []
    for r in final_runs[agent]:
        net = make_net(agent, r["cfg"])
        net.load_state_dict(r["log"].best_state)
        net.eval()
        pols.append(TorchPolicy(net, agent))
    return pols

POLICIES = {agent: trained_policies(agent) for agent in TRAINERS}
CONVENTIONAL = {"Rule-based": rule, "Lookahead (2-step)": lookahead}
METRICS = ["objective", "energy_cost", "discomfort_degh", "violation_hours", "peak_import_kw",
           "exceed_kwh", "exceed_hours", "throughput_kwh", "final_soc"]

def evaluate_all(day_set, cfg=BASE_CFG):
    """returns {controller: [per-day DataFrame for each seed]} (one entry for conventional controllers)"""
    out = {}
    for name, c in CONVENTIONAL.items():
        lk = LookaheadController(cfg) if name.startswith("Lookahead") else c
        out[name] = [run_controller(lk, day_set, cfg)]
    for agent, pols in POLICIES.items():
        out[agent] = [run_controller(p, day_set, cfg) for p in pols]
    return out

def results_table(res):
    rows = {}
    for name, dfs in res.items():
        per_seed = pd.DataFrame([summarise(df) for df in dfs])
        row = {}
        for m in METRICS:
            mu, sd = per_seed[m].mean(), per_seed[m].std(ddof=0)
            row[m] = f"{mu:.2f}" if len(dfs) == 1 else f"{mu:.2f} ± {sd:.2f}"
        row["latency_ms"] = f"{np.mean([df.attrs['latency_ms'] for df in dfs]):.3f}"
        rows[name] = row
    return pd.DataFrame(rows).T

def mean_metric(res, name, metric):
    return np.mean([summarise(df)[metric] for df in res[name]])

t0 = time.perf_counter()
E1 = evaluate_all(test_days)
print(f"evaluation took {time.perf_counter() - t0:.0f} s")
e1_table = results_table(E1)
e1_table.to_csv(os.path.join(RESULTS_DIR, "E1_test_days.csv"))
e1_table

Units: objective, energy cost in \$/day; discomfort in °C·h/day; violation and exceed hours per day; peak import in kW (daily maximum, averaged over days); exceedance and throughput in kWh/day; final SoC as a fraction.

The daily objective is the quantity every agent was trained to minimise, so it is the headline number. The other columns show where each controller's cost comes from. We report them separately so that, for example, a low bill cannot hide a hot office.

In [ ]:
names = list(E1.keys())
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
data = [np.concatenate([df["objective"].values for df in E1[n]]) for n in names]
axes[0].boxplot(data, showfliers=False)
axes[0].set_xticks(range(1, len(names) + 1), [n.replace(" (2-step)", "") for n in names], rotation=20)
axes[0].set(ylabel="daily objective ($)", title="E1: distribution over test days (all seeds)")

comp = pd.DataFrame({n: [mean_metric(E1, n, "energy_cost"),
                         BASE_CFG.lambda_d * mean_metric(E1, n, "discomfort_degh"),
                         BASE_CFG.lambda_p * mean_metric(E1, n, "exceed_kwh"),
                         BASE_CFG.lambda_b * mean_metric(E1, n, "throughput_kwh")] for n in names},
                    index=["energy bill", "discomfort penalty", "peak penalty", "battery wear"]).T
comp.plot.bar(stacked=True, ax=axes[1], rot=20)
axes[1].set(ylabel="$/day", title="Where the objective comes from")
axes[1].legend(fontsize=7)

hot = test_days.temp.max(1) >= 35
for n in names:
    obj = np.mean([df["objective"].values for df in E1[n]], axis=0)
    axes[2].scatter(test_days.temp.max(1), obj, s=8, label=n.replace(" (2-step)", ""))
axes[2].set(xlabel="daily max outdoor temp (°C)", ylabel="daily objective ($)", title="Cost vs heat")
axes[2].legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "E1_test_days.png"), dpi=150)
plt.show()

Since the RL agents never saw these days, we also split the test days into ordinary and hot days (daily maximum of 35 °C or more). This shows whether the ranking holds up under the conditions that matter most for peak demand.

In [ ]:
split_rows = {}
for n in names:
    obj = np.mean([df["objective"].values for df in E1[n]], axis=0)
    peak = np.mean([df["peak_import_kw"].values for df in E1[n]], axis=0)
    disc = np.mean([df["discomfort_degh"].values for df in E1[n]], axis=0)
    split_rows[n] = {"objective (<35C days)": obj[~hot].mean(), "objective (>=35C days)": obj[hot].mean(),
                     "peak kW (>=35C days)": peak[hot].mean(), "discomfort (>=35C days)": disc[hot].mean()}
print(f"{hot.sum()} of {len(hot)} test days reached 35 C or more")
pd.DataFrame(split_rows).T.round(2)

## 14. Experiment E2: shifted occupancy schedule
**Author: Priyandan Francis**

Same test days, but the office now opens at 10:00 and closes at 20:00 (occupancy and the linked non-cooling load shifted by 2 hours). Nothing is retrained. The agents see the new schedule only through the current and next-hour occupancy and load in the observation. Because the tariff is unchanged, the occupied period now overlaps more with the 15:00-21:00 peak. This makes E2 a test of whether the agents learned to react to occupancy, or just memorised the time of day.

In [ ]:
cfg_e2 = replace(BASE_CFG, schedule_shift=2)
E2 = evaluate_all(test_days, cfg_e2)
e2_table = results_table(E2)
e2_table.to_csv(os.path.join(RESULTS_DIR, "E2_shifted_schedule.csv"))
e2_table

## 15. Experiment E3: weather stress tests
**Author: Priyandan Francis**

Two predefined cases applied to all test days, again without retraining:
- **High heat:** outdoor temperature +4 °C every hour (also lowers PV output a little through the cell-temperature term).
- **Low solar:** irradiance × 0.5, e.g. a smoky or cloudy summer. This halves PV and also reduces solar heat gain.

Both push the observation outside the range seen in training. They are bounded case studies of robustness, not proof that the agents would transfer to a different building.

In [ ]:
cfg_hot = replace(BASE_CFG, temp_offset=4.0)
cfg_dim = replace(BASE_CFG, solar_scale=0.5)
E3_hot = evaluate_all(test_days, cfg_hot)
E3_dim = evaluate_all(test_days, cfg_dim)
e3_hot_table, e3_dim_table = results_table(E3_hot), results_table(E3_dim)
e3_hot_table.to_csv(os.path.join(RESULTS_DIR, "E3_high_heat.csv"))
e3_dim_table.to_csv(os.path.join(RESULTS_DIR, "E3_low_solar.csv"))
print("High heat (+4 C)")
display(e3_hot_table[["objective", "energy_cost", "discomfort_degh", "peak_import_kw", "exceed_kwh"]])
print("Low solar (x0.5)")
display(e3_dim_table[["objective", "energy_cost", "discomfort_degh", "peak_import_kw", "exceed_kwh"]])

Summary across the four scenarios. Each bar is the mean daily objective, and the error bar is the standard deviation across RL seeds.

In [ ]:
scenarios = {"E1 standard": E1, "E2 shifted schedule": E2, "E3 high heat": E3_hot, "E3 low solar": E3_dim}
fig, ax = plt.subplots(figsize=(11, 3.6))
width = 0.8 / len(names)
for i, n in enumerate(names):
    means = [mean_metric(res, n, "objective") for res in scenarios.values()]
    sds = [np.std([summarise(df)["objective"] for df in res[n]]) for res in scenarios.values()]
    ax.bar(np.arange(len(scenarios)) + i * width, means, width, yerr=sds, capsize=2, label=n)
ax.set_xticks(np.arange(len(scenarios)) + width * (len(names) - 1) / 2, list(scenarios.keys()))
ax.set(ylabel="mean daily objective ($, lower is better)", title="All controllers across scenarios")
ax.legend(fontsize=8, ncol=5)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "scenario_comparison.png"), dpi=150)
plt.show()

scenario_summary = pd.DataFrame({s: {n: mean_metric(res, n, "objective") for n in names}
                                 for s, res in scenarios.items()})
scenario_summary.to_csv(os.path.join(RESULTS_DIR, "scenario_summary.csv"))
scenario_summary.round(2)

## 16. 24-hour demonstration day
**Author: Dasith Gamage**

For the presentation we replay one unseen day under every controller: the hottest weekday in the test set. For each RL agent we use the seed whose checkpoint had the best validation objective. Everything plotted is measured simulator output from these runs.

In [ ]:
wd_idx = np.where(test_days.weekday)[0]
demo_day = int(wd_idx[np.argmax(test_days.temp.max(1)[wd_idx])])
print("demo day:", test_days.dates[demo_day].date(), "max temp", test_days.temp[demo_day].max(), "C")

demo_ctrl = dict(CONVENTIONAL)
for agent in TRAINERS:
    best_seed = int(np.argmin([r["log"].best_objective for r in final_runs[agent]]))
    demo_ctrl[agent] = POLICIES[agent][best_seed]
demo = {n: run_controller(c if not n.startswith("Lookahead") else LookaheadController(),
                          test_days, days=[demo_day], keep_trace=True) for n, c in demo_ctrl.items()}

fig, axes = plt.subplots(len(demo), 4, figsize=(15, 2.3 * len(demo)), sharex=True)
hrs = np.arange(24)
for row, (n, df) in enumerate(demo.items()):
    tr = df.attrs["trace"]
    ax = axes[row, 0]
    ax.axhspan(BASE_CFG.t_low, BASE_CFG.t_high, color="green", alpha=0.1)
    ax.fill_between(hrs, 18, 34, where=tr["occ"] > 0, step="post", color="grey", alpha=0.1)
    ax.plot(hrs, tr["Ta"], label="indoor air")
    ax.plot(hrs, tr["Tm"], label="thermal mass", ls="--")
    ax.set(ylim=(18, 34), ylabel=f"{n}\n°C")
    axes[row, 1].step(hrs, tr["p_cool"], where="post", c="C3")
    axes[row, 1].set(ylim=(-1, 13), ylabel="cooling kW")
    axes[row, 2].plot(hrs, tr["soc"] * 100, c="C4")
    axes[row, 2].set(ylim=(0, 100), ylabel="SoC %")
    axes[row, 3].step(hrs, tr["p_import"], where="post", label="grid import")
    axes[row, 3].plot(hrs, tr["p_pv"], c="orange", label="PV")
    axes[row, 3].axhline(BASE_CFG.peak_limit, c="r", ls=":", lw=1)
    axes[row, 3].set(ylim=(0, 45), ylabel="kW")
    axes[row, 3].text(0.3, 38, f"${df['objective'].iloc[0]:.1f} objective, "
                               f"{df['energy_cost'].iloc[0]:.1f} bill", fontsize=7)
axes[0, 0].legend(fontsize=7, loc="upper left")
axes[0, 3].legend(fontsize=7, loc="upper right")
axes[0, 0].set_title("Temperatures (green = comfort band, grey = occupied)")
axes[0, 1].set_title("Cooling (electric)")
axes[0, 2].set_title("Battery state of charge")
axes[0, 3].set_title("Grid import vs PV (red = 15 kW limit)")
for ax in axes[-1]:
    ax.set_xlabel("hour of day")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "demo_day.png"), dpi=150)
plt.show()

pd.DataFrame({n: df[METRICS].iloc[0] for n, df in demo.items()}).T.round(2)

## 17. Summary of findings
**Author: all members**

*To be written by the group after the full run, using the numbers printed above.* Suggested points to cover:
- E1: which controllers beat the rule-based controller on the test days, and by how much (objective, bill, peak import, discomfort). How close each agent gets to the two-step lookahead.
- Learning quality: seed-to-seed spread, steps needed to beat the rule-based level, and training time (Section 12 table).
- Tuning: effect of the learning rate, Double DQN, the REINFORCE baseline, and the A2C rollout length (Section 11). Our pilot runs found that short bootstrapped A2C rollouts stall around the rule-based level.
- E2/E3: which agents degrade gracefully under the shifted schedule and the weather stress tests, and which ones don't.
- Limitations to carry into the report: no calibration against a real building, fixed initial temperatures each day, 24-hour horizon with a terminal battery adjustment only, persistence forecast for the lookahead, and only 3 seeds.

## 18. References

[1] Z. Wang and T. Hong, "Reinforcement learning for building controls: The opportunities and challenges," *Applied Energy*, vol. 269, art. 115036, 2020, doi: 10.1016/j.apenergy.2020.115036.

[2] P. Bacher and H. Madsen, "Identifying suitable models for the heat dynamics of buildings," *Energy and Buildings*, vol. 43, no. 7, pp. 1511-1522, 2011, doi: 10.1016/j.enbuild.2011.02.005.

[3] V. Mnih *et al.*, "Human-level control through deep reinforcement learning," *Nature*, vol. 518, pp. 529-533, 2015, doi: 10.1038/nature14236.

[4] R. J. Williams, "Simple statistical gradient-following algorithms for connectionist reinforcement learning," *Machine Learning*, vol. 8, pp. 229-256, 1992, doi: 10.1007/BF00992696.

[5] V. Mnih *et al.*, "Asynchronous methods for deep reinforcement learning," in *Proc. 33rd Int. Conf. Machine Learning (ICML)*, PMLR vol. 48, pp. 1928-1937, 2016.

[6] Open-Meteo, "Historical Weather API." [Online]. Available: https://open-meteo.com/en/docs/historical-weather-api

[7] H. van Hasselt, A. Guez, and D. Silver, "Deep reinforcement learning with double Q-learning," in *Proc. AAAI Conf. Artificial Intelligence*, vol. 30, no. 1, 2016, doi: 10.1609/aaai.v30i1.10295.

[8] R. S. Sutton and A. G. Barto, *Reinforcement Learning: An Introduction*, 2nd ed. Cambridge, MA, USA: MIT Press, 2018.